# Avatar_ML Colab inference server

Three cells. Run them in order, top to bottom. Each one fails loudly if something breaks (no silent rollbacks).

| Cell | What it does | Time |
|---|---|---|
| **Cell 1 — Setup** | Installs Python 3.10, all model deps, clones OpenVoice / MeloTTS / MuseTalk, downloads weights. 11 numbered steps. Halts on first error. | ~10 min |
| **Cell 2 — server.py** | Writes the FastAPI inference server to `/content/server.py`. Instant. | <1 s |
| **Cell 3 — Launch** | Pre-flights the venv, starts cloudflared + uvicorn, verifies `/healthz`, prints `COLAB_INFERENCE_URL=...` once verified. | ~30 s |

## Setup

1. **Runtime → Change runtime type → T4 GPU** → Save.
2. Run **Cell 1** (wait for `✓ ALL SETUP COMPLETE`).
3. Run **Cell 2** (instant — writes the server file).
4. Run **Cell 3** (~30 s — prints `READY` + a tunnel URL).
5. Copy the `COLAB_INFERENCE_URL=https://...trycloudflare.com` line into your local `.env` on Windows.

## If something goes wrong

Each cell prints `Step N: <name>` banners and halts on the first failing step with the actual error. Paste that into the chat and we'll fix the one specific line.

**To restart the server** (e.g. after editing server.py): re-run **Cell 3** only. It kills the old uvicorn/cloudflared and starts fresh.

**After a Colab disconnect**: re-run all three cells from the top. Cell 1's idempotency means already-installed deps are skipped.

In [ ]:
# Cell 1 — Setup. Run this ONCE per Colab session. Takes ~10 minutes.
#
# This installs Python 3.10, all model deps, clones repos, and downloads
# weights. Every step uses subprocess with check=True, so the cell HALTS
# immediately on any error and prints the actual failure — no silent rollbacks.
#
# Re-running is safe: already-installed steps are skipped.

import os, sys, subprocess
print(f"Notebook kernel: {sys.version.split()[0]}")
subprocess.run(['nvidia-smi'], check=False)

PY310 = '/content/py310'
PY  = f'{PY310}/bin/python'
PIP = f'{PY310}/bin/pip'


def run(cmd, check=True, shell=False):
    """Run a command, print its output, raise on failure (if check=True)."""
    label = cmd if isinstance(cmd, str) else ' '.join(cmd)
    print(f"\n$ {label}", flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True, shell=shell)
    if r.stdout:
        print(r.stdout, end='' if r.stdout.endswith('\n') else '\n')
    if r.stderr:
        print(r.stderr, end='' if r.stderr.endswith('\n') else '\n')
    if check and r.returncode != 0:
        raise RuntimeError(f"Step failed (exit {r.returncode}). Fix the error above and re-run this cell.")
    return r


def step(n, title, fn):
    bar = '=' * 70
    print(f"\n{bar}\n  Step {n}: {title}\n{bar}", flush=True)
    fn()
    print(f"  ✓ Step {n} done.")


# ─── Step 1: Python 3.10 ────────────────────────────────────────────────────
def install_python310():
    if subprocess.run(['which', 'python3.10'], capture_output=True).returncode == 0:
        print('python3.10 already installed.')
    else:
        run(['sudo', 'apt-get', 'update', '-qq'])
        r = subprocess.run(
            ['sudo', 'apt-get', 'install', '-y', '-qq',
             'python3.10', 'python3.10-venv', 'python3.10-dev'],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            print('Direct apt failed; falling back to deadsnakes PPA...')
            run(['sudo', 'apt-get', 'install', '-y', '-qq', 'software-properties-common'])
            run(['sudo', 'add-apt-repository', '-y', 'ppa:deadsnakes/ppa'])
            run(['sudo', 'apt-get', 'update', '-qq'])
            run(['sudo', 'apt-get', 'install', '-y', '-qq',
                 'python3.10', 'python3.10-venv', 'python3.10-dev'])
    run(['python3.10', '--version'])


# ─── Step 2: Venv ───────────────────────────────────────────────────────────
def create_venv():
    if os.path.exists(f'{PY310}/bin/python'):
        print(f'venv already exists at {PY310}.')
    else:
        run(['python3.10', '-m', 'venv', PY310])
    run([PIP, 'install', '--upgrade', 'pip'])


# ─── Step 3: Server runtime ─────────────────────────────────────────────────
def install_server_runtime():
    run([PIP, 'install',
         'fastapi>=0.115',
         'uvicorn[standard]>=0.32',
         'python-multipart>=0.0.12',
         'websockets>=13',
         'Pillow>=10.4',
         'soundfile>=0.12',
         'librosa==0.9.1',
         'huggingface_hub>=0.24',
         'numpy==1.26.4',
         'opencv-python==4.8.1.78'])


# ─── Step 4: Torch (CUDA 12.1 for T4) ───────────────────────────────────────
def install_torch():
    run([PIP, 'install',
         'torch==2.1.2', 'torchaudio==2.1.2', 'torchvision==0.16.2',
         '--index-url', 'https://download.pytorch.org/whl/cu121'])


# ─── Step 5: Clone repos ────────────────────────────────────────────────────
def clone_repos():
    for name, url in [
        ('OpenVoice', 'https://github.com/myshell-ai/OpenVoice'),
        ('MeloTTS',   'https://github.com/myshell-ai/MeloTTS'),
        ('MuseTalk',  'https://github.com/TMElyralab/MuseTalk'),
    ]:
        if os.path.exists(f'/content/{name}'):
            print(f'{name} already cloned.')
        else:
            run(['git', 'clone', '--depth', '1', url, f'/content/{name}'])


# ─── Step 6: Voice/TTS deps ─────────────────────────────────────────────────
def install_voice_deps():
    # OpenVoice runtime deps
    run([PIP, 'install',
         'Unidecode>=1.3', 'openai-whisper', 'whisper-timestamped',
         'faster-whisper', 'pydub', 'wavmark', 'inflect', 'eng-to-ipa',
         'pypinyin', 'cn2an', 'jieba', 'langid', 'dominate'])
    # MeloTTS runtime deps
    run([PIP, 'install',
         'mecab-python3>=1.0', 'unidic-lite>=1.0', 'num2words',
         'pykakasi', 'fugashi', 'g2p_en', 'anyascii', 'jamo',
         'gruut', 'cached_path', 'nltk',
         'transformers==4.27.4', 'tokenizers==0.13.3'])
    # .pth files so the venv can `import openvoice`, `import melo`, `import musetalk`
    sp = f'{PY310}/lib/python3.10/site-packages'
    for repo in ['OpenVoice', 'MeloTTS', 'MuseTalk']:
        run(f"echo '/content/{repo}' > {sp}/{repo.lower()}.pth", shell=True)
    # MeloTTS needs the unidic dictionary
    run([PY, '-m', 'unidic', 'download'], check=False)


# ─── Step 7: Voice import sanity check ──────────────────────────────────────
def voice_sanity():
    run([PY, '-c',
         'from unidecode import unidecode; '
         'import cv2; '
         'from openvoice.api import ToneColorConverter; '
         'from openvoice import se_extractor; '
         'import whisper; '
         'from melo.api import TTS; '
         'print("voice imports OK")'])


# ─── Step 8: OpenVoice V2 weights ───────────────────────────────────────────
def download_openvoice_weights():
    ckpt = '/content/models/openvoice/checkpoints_v2'
    os.makedirs(ckpt, exist_ok=True)
    run([PY, '-c',
         f"from huggingface_hub import snapshot_download; "
         f"snapshot_download(repo_id='myshell-ai/OpenVoiceV2', local_dir='{ckpt}')"])


# ─── Step 9: MuseTalk deps (chumpy + mmpose) ────────────────────────────────
def install_musetalk_deps():
    # chumpy needs setuptools<70 (uses removed distutils path in newer setuptools)
    run([PIP, 'install', 'setuptools<70', 'wheel'])
    run([PIP, 'install', 'chumpy'])
    run([PIP, 'install', 'openmim'])
    run([PY, '-m', 'mim', 'install', 'mmcv==2.1.0'])
    run([PY, '-m', 'mim', 'install', 'mmdet==3.2.0'])
    run([PY, '-m', 'mim', 'install', 'mmpose==1.3.1'])
    # MuseTalk requirements.txt may have extras (gradio, diffusers); non-critical.
    run([PIP, 'install', '-r', '/content/MuseTalk/requirements.txt'], check=False)


# ─── Step 10: MuseTalk weights ──────────────────────────────────────────────
def download_musetalk_weights():
    run('cd /content/MuseTalk && bash download_weights.sh', shell=True, check=False)


# ─── Step 11: Final MuseTalk import sanity ──────────────────────────────────
def final_sanity():
    run([PY, '-c',
         'import chumpy; '
         'from mmpose.apis import init_model; '
         'from musetalk.utils.utils import load_all_model; '
         'from musetalk.utils.preprocessing import get_landmark_and_bbox; '
         'from musetalk.utils.blending import get_image_prepare_material, get_image; '
         'print("MuseTalk imports OK")'])


# ─── Run all steps in order ─────────────────────────────────────────────────
step(1,  'Install Python 3.10',                          install_python310)
step(2,  'Create /content/py310 venv',                   create_venv)
step(3,  'Install server runtime (fastapi, uvicorn, cv2)', install_server_runtime)
step(4,  'Install torch for CUDA 12.1',                  install_torch)
step(5,  'Clone OpenVoice / MeloTTS / MuseTalk repos',   clone_repos)
step(6,  'Install OpenVoice + MeloTTS runtime deps',     install_voice_deps)
step(7,  'Sanity-check voice imports',                   voice_sanity)
step(8,  'Download OpenVoice V2 weights',                download_openvoice_weights)
step(9,  'Install MuseTalk deps (chumpy + mmpose)',      install_musetalk_deps)
step(10, 'Download MuseTalk weights',                    download_musetalk_weights)
step(11, 'Final MuseTalk import sanity',                 final_sanity)

print('\n' + '=' * 70)
print('✓ ALL SETUP COMPLETE.')
print('  Next: run Cell 2 (writes server.py), then Cell 3 (launches uvicorn).')
print('=' * 70)

In [ ]:
%%writefile /content/server.py
"""Avatar_ML Colab inference server. Runs under /content/py310/bin/python.

Wire protocol mirrors Plan.md §5.5. Do not change message shapes without also
updating the Windows-side ColabWorkerClient.
"""
import os, sys, io, json, base64, time, tarfile, shutil, traceback, pickle, glob
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
import torch
import cv2
from fastapi import FastAPI, UploadFile, File, Request, Query, WebSocket, WebSocketDisconnect, HTTPException
from fastapi.responses import Response

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OPENVOICE_CKPT = '/content/models/openvoice/checkpoints_v2'
MUSETALK_DIR  = '/content/MuseTalk'
CACHE_ROOT    = '/content/cache'
Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)
print('[server] Python:', sys.version.split()[0], '| Device:', DEVICE)

if MUSETALK_DIR not in sys.path:
    sys.path.insert(0, MUSETALK_DIR)

# ============================================================================
# Lazy loaders
# ============================================================================
_ov = {'tcc': None, 'melo': None, 'source_se': None}
_mt = {'loaded': False, 'modules': {}}

def lazy_load_openvoice():
    if _ov['tcc'] is not None:
        return _ov
    from openvoice.api import ToneColorConverter
    tcc = ToneColorConverter(f'{OPENVOICE_CKPT}/converter/config.json', device=DEVICE)
    tcc.load_ckpt(f'{OPENVOICE_CKPT}/converter/checkpoint.pth')
    _ov['tcc'] = tcc
    from melo.api import TTS
    _ov['melo'] = TTS(language='EN', device=DEVICE)
    _ov['source_se'] = torch.load(
        f'{OPENVOICE_CKPT}/base_speakers/ses/en-us.pth', map_location=DEVICE
    )
    print('[lazy] OpenVoice V2 + MeloTTS loaded.')
    return _ov

def lazy_load_musetalk():
    if _mt['loaded']:
        return _mt
    from musetalk.utils.utils import load_all_model
    audio_processor, vae, unet, pe = load_all_model()
    vae.vae = vae.vae.half().to(DEVICE)
    unet.model = unet.model.half().to(DEVICE)
    pe = pe.half().to(DEVICE)
    _mt['modules'] = {
        'audio_processor': audio_processor, 'vae': vae, 'unet': unet, 'pe': pe,
    }
    _mt['loaded'] = True
    print('[lazy] MuseTalk loaded.')
    return _mt

# ============================================================================
# Defensive shims around MuseTalk's shifting API
# ============================================================================
def _vae_encode(vae, crop_256):
    """VAE-encode a 256x256 face crop. Method name varies across MuseTalk versions."""
    for name in ('get_latents_for_unet', 'encode_latents', 'encode'):
        fn = getattr(vae, name, None)
        if fn is not None:
            return fn(crop_256)
    raise AttributeError(
        f'VAE has no known encode method. Tried get_latents_for_unet/encode_latents/encode. '
        f'Available: {[m for m in dir(vae) if not m.startswith("_")][:30]}'
    )

def _vae_decode(vae, pred_latents):
    """Decode predicted latents to face crops."""
    for name in ('decode_latents', 'decode'):
        fn = getattr(vae, name, None)
        if fn is not None:
            return fn(pred_latents)
    raise AttributeError(
        f'VAE has no known decode method. Available: {[m for m in dir(vae) if not m.startswith("_")][:30]}'
    )

def _get_image_material(frame, bbox):
    """Normalize get_image_prepare_material return to (mask, crop_box)."""
    from musetalk.utils.blending import get_image_prepare_material
    res = get_image_prepare_material(frame, bbox)
    if isinstance(res, tuple) and len(res) == 2:
        return res
    return res, bbox

def _blend(frame, face, bbox, mask, crop_box):
    """Call get_image with either 5-arg or 4-arg signature."""
    from musetalk.utils.blending import get_image
    try:
        return get_image(frame, face, bbox, mask, crop_box)
    except TypeError:
        return get_image(frame, face, bbox, mask)

def _is_placeholder(bbox):
    """Works for tuple, list, numpy array — coord_placeholder is (0,0,0,0)."""
    try:
        return all(int(x) == 0 for x in bbox)
    except Exception:
        from musetalk.utils.preprocessing import coord_placeholder
        return bbox == coord_placeholder

def _unet_forward(unet, latent_batch, audio_feat):
    """unet.model(...) may return tensor directly or an object with .sample."""
    out = unet.model(latent_batch, 0, encoder_hidden_states=audio_feat)
    return getattr(out, 'sample', out)

def _feature2chunks(audio_processor, feature_array, fps):
    """feature2chunks kwargs vary across MuseTalk versions."""
    try:
        return audio_processor.feature2chunks(feature_array=feature_array, fps=fps)
    except TypeError:
        return audio_processor.feature2chunks(feature_array, fps)

def _unet_dtype(unet):
    """Best-effort: figure out the dtype of the UNet model for casting inputs."""
    try:
        return unet.model.dtype
    except AttributeError:
        try:
            return next(unet.model.parameters()).dtype
        except Exception:
            return torch.float16

# ============================================================================
# FastAPI app
# ============================================================================
app = FastAPI(title='Avatar_ML Colab inference server')

@app.get('/healthz')
def healthz():
    free_vram_mb = None
    if torch.cuda.is_available():
        free, _ = torch.cuda.mem_get_info()
        free_vram_mb = int(free / 1e6)
    return {
        'status': 'ok',
        'python': sys.version.split()[0],
        'gpu': torch.cuda.is_available(),
        'openvoice_loaded': _ov['tcc'] is not None,
        'musetalk_loaded': _mt['loaded'],
        'free_vram_mb': free_vram_mb,
    }

# ============================================================================
# Voice — fully implemented
# ============================================================================
@app.post('/voice/extract_embedding')
async def extract_voice_embedding(audio: UploadFile = File(...)):
    try:
        ov = lazy_load_openvoice()
        upload_dir = Path('/content/uploads/voice')
        upload_dir.mkdir(parents=True, exist_ok=True)
        wav_path = upload_dir / (audio.filename or 'voice.wav')
        wav_path.write_bytes(await audio.read())
        from openvoice import se_extractor
        target_se, _name = se_extractor.get_se(str(wav_path), ov['tcc'], vad=False)
        buf = io.BytesIO()
        np.save(buf, target_se.detach().cpu().numpy())
        return Response(content=buf.getvalue(), media_type='application/octet-stream')
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f'{type(e).__name__}: {e}')

# ============================================================================
# Avatar preprocessing
# ============================================================================
def musetalk_preprocess(video_path: Path, out_dir: Path, bbox_shift: int = 0):
    """Run MuseTalk preprocessing on video_path. Writes artifacts into out_dir/.

    Output layout:
      out_dir/coords.pkl
      out_dir/mask_coords.pkl
      out_dir/latents.pt
      out_dir/full_imgs/NNNNNNNN.png
      out_dir/mask/NNNNNNNN.png
      out_dir/avatar_info.json
    """
    from musetalk.utils.preprocessing import get_landmark_and_bbox
    from musetalk.utils.utils import get_video_fps

    mt = lazy_load_musetalk()
    vae = mt['modules']['vae']

    out_dir = Path(out_dir)
    full_imgs_dir = out_dir / 'full_imgs'
    mask_dir = out_dir / 'mask'
    out_dir.mkdir(parents=True, exist_ok=True)
    full_imgs_dir.mkdir(exist_ok=True)
    mask_dir.mkdir(exist_ok=True)

    fps = get_video_fps(str(video_path))
    print(f'[preprocess] video fps={fps}')

    cap = cv2.VideoCapture(str(video_path))
    img_paths = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        p = str(full_imgs_dir / f'{idx:08d}.png')
        cv2.imwrite(p, frame)
        img_paths.append(p)
        idx += 1
    cap.release()
    print(f'[preprocess] extracted {len(img_paths)} frames -> {full_imgs_dir}')

    print(f'[preprocess] running face detection (slow part)...')
    coord_list, frame_list = get_landmark_and_bbox(img_paths, upperbondrange=bbox_shift)
    n_detected = sum(1 for b in coord_list if not _is_placeholder(b))
    print(f'[preprocess] detected faces in {n_detected}/{len(coord_list)} frames')

    print(f'[preprocess] VAE-encoding {n_detected} face crops...')
    input_latent_list = []
    for bbox, frame in zip(coord_list, frame_list):
        if _is_placeholder(bbox):
            continue
        x1, y1, x2, y2 = bbox
        crop_frame = frame[y1:y2, x1:x2]
        crop_frame = cv2.resize(crop_frame, (256, 256), interpolation=cv2.INTER_LANCZOS4)
        latent = _vae_encode(vae, crop_frame)
        input_latent_list.append(latent)
    print(f'[preprocess] encoded {len(input_latent_list)} face latents')

    print(f'[preprocess] generating masks...')
    mask_coords_list = []
    for i, (bbox, frame) in enumerate(zip(coord_list, frame_list)):
        if _is_placeholder(bbox):
            mask_coords_list.append((0, 0, 0, 0))
            continue
        mask, crop_box = _get_image_material(frame, bbox)
        cv2.imwrite(str(mask_dir / f'{i:08d}.png'), mask)
        mask_coords_list.append(crop_box)
    print(f'[preprocess] wrote {len(os.listdir(mask_dir))} masks')

    with open(out_dir / 'coords.pkl', 'wb') as f:
        pickle.dump(coord_list, f)
    with open(out_dir / 'mask_coords.pkl', 'wb') as f:
        pickle.dump(mask_coords_list, f)
    torch.save(input_latent_list, out_dir / 'latents.pt')
    with open(out_dir / 'avatar_info.json', 'w') as f:
        json.dump({'bbox_shift': bbox_shift, 'fps': fps,
                   'num_frames': len(img_paths)}, f)
    print(f'[preprocess] done')

@app.post('/avatar/preprocess')
async def preprocess_avatar(video: UploadFile = File(...)):
    try:
        upload_dir = Path('/content/uploads/avatar')
        upload_dir.mkdir(parents=True, exist_ok=True)
        mp4_path = upload_dir / (video.filename or 'face.mp4')
        mp4_path.write_bytes(await video.read())

        ts = int(time.time())
        tmp_root = Path(CACHE_ROOT) / f'_preproc_{ts}'
        cache_dir = tmp_root / 'cache'
        musetalk_preprocess(mp4_path, cache_dir)

        tar_path = Path(f'/tmp/avatar_cache_{ts}.tar.gz')
        with tarfile.open(tar_path, 'w:gz') as tar:
            tar.add(cache_dir, arcname='cache')
        body = tar_path.read_bytes()
        tar_path.unlink(missing_ok=True)
        shutil.rmtree(tmp_root, ignore_errors=True)

        return Response(content=body, media_type='application/gzip')
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f'{type(e).__name__}: {e}')

@app.post('/avatar/upload_cache')
async def upload_avatar_cache(request: Request, avatar_id: str = Query(...)):
    body = await request.body()
    dest = Path(CACHE_ROOT) / avatar_id
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    tar_path = dest / 'cache.tar.gz'
    tar_path.write_bytes(body)
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(dest)
    return {'avatar_id': avatar_id, 'extracted_to': str(dest), 'size_bytes': len(body)}

# ============================================================================
# TTS — fully implemented
# ============================================================================
def _to_24khz_int16(wav_path: str) -> bytes:
    data, sr = sf.read(wav_path, dtype='int16')
    if data.ndim > 1:
        data = data[:, 0]
    if sr != 24000:
        f = data.astype(np.float32) / 32768.0
        f = librosa.resample(f, orig_sr=sr, target_sr=24000)
        data = np.clip(f * 32768.0, -32768, 32767).astype(np.int16)
    return data.tobytes()

def _synthesize_pcm(text: str, target_se_b64: str) -> bytes:
    ov = lazy_load_openvoice()
    arr = np.load(io.BytesIO(base64.b64decode(target_se_b64)))
    target_se = torch.from_numpy(arr).to(DEVICE)
    if target_se.dim() == 1:
        target_se = target_se.unsqueeze(0).unsqueeze(-1)
    src_path = '/tmp/openvoice_src.wav'
    out_path = '/tmp/openvoice_out.wav'
    spk_ids = ov['melo'].hps.data.spk2id
    spk_key = ('EN-US' if 'EN-US' in spk_ids
               else 'EN-Default' if 'EN-Default' in spk_ids
               else next(iter(spk_ids)))
    ov['melo'].tts_to_file(text, spk_ids[spk_key], src_path, speed=1.0)
    ov['tcc'].convert(
        audio_src_path=src_path,
        src_se=ov['source_se'],
        tgt_se=target_se,
        output_path=out_path,
    )
    return _to_24khz_int16(out_path)

@app.websocket('/tts/stream')
async def tts_stream(ws: WebSocket):
    await ws.accept()
    try:
        payload = json.loads(await ws.receive_text())
        pcm_bytes = _synthesize_pcm(payload['text'], payload['embedding_b64'])
        chunk_sz = 9600
        pts = 0.0
        for i in range(0, len(pcm_bytes), chunk_sz):
            chunk = pcm_bytes[i:i + chunk_sz]
            await ws.send_text(json.dumps({
                'type': 'pcm', 'pts': pts,
                'data': base64.b64encode(chunk).decode(),
            }))
            pts += (len(chunk) // 2) / 24000.0
        await ws.send_text(json.dumps({'type': 'end'}))
    except WebSocketDisconnect:
        pass
    except Exception as e:
        traceback.print_exc()
        try:
            await ws.send_text(json.dumps({'type': 'error', 'message': f'{type(e).__name__}: {e}'}))
        except Exception:
            pass

# ============================================================================
# Lip-sync
# ============================================================================
def _ping_pong_index(i: int, n: int) -> int:
    if n <= 1:
        return 0
    period = 2 * (n - 1)
    i = i % period
    return i if i < n else period - i

def _run_musetalk(avatar_id: str, pcm_bytes: bytes, fps: int = 25):
    """Generator yielding (frame_idx, jpeg_bytes) for the full audio."""
    from musetalk.utils.utils import datagen
    from musetalk.utils.preprocessing import read_imgs

    mt = lazy_load_musetalk()
    audio_processor = mt['modules']['audio_processor']
    vae    = mt['modules']['vae']
    unet   = mt['modules']['unet']
    pe     = mt['modules']['pe']

    cache_dir = Path(CACHE_ROOT) / avatar_id / 'cache'
    if not cache_dir.exists():
        raise FileNotFoundError(
            f'No avatar cache at {cache_dir}. Re-upload via /avatar/upload_cache.'
        )

    with open(cache_dir / 'coords.pkl', 'rb') as f:
        coord_list = pickle.load(f)
    with open(cache_dir / 'mask_coords.pkl', 'rb') as f:
        mask_coords_list = pickle.load(f)
    input_latent_list = torch.load(cache_dir / 'latents.pt')

    full_imgs_paths = sorted(glob.glob(str(cache_dir / 'full_imgs' / '*.png')))
    mask_paths      = sorted(glob.glob(str(cache_dir / 'mask' / '*.png')))
    frame_list = read_imgs(full_imgs_paths)
    mask_list  = read_imgs(mask_paths)
    print(f'[lipsync] loaded cache: {len(frame_list)} frames, '
          f'{len(input_latent_list)} latents')

    # OpenVoice outputs 24 kHz; Whisper (inside audio_processor) expects 16 kHz.
    # Explicitly resample here so we don't depend on audio_processor handling it.
    audio_path = '/tmp/lipsync_input.wav'
    arr = np.frombuffer(pcm_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    arr_16k = librosa.resample(arr, orig_sr=24000, target_sr=16000)
    sf.write(audio_path, (arr_16k * 32768.0).clip(-32768, 32767).astype(np.int16),
             16000, subtype='PCM_16')

    print('[lipsync] extracting whisper features...')
    whisper_feature = audio_processor.audio2feat(audio_path)
    whisper_chunks  = _feature2chunks(audio_processor, whisper_feature, fps)
    print(f'[lipsync] {len(whisper_chunks)} audio chunks')

    n_latents = len(input_latent_list)
    batch_size = 8
    gen = datagen(
        whisper_chunks=whisper_chunks,
        vae_encode_latents=input_latent_list,
        batch_size=batch_size,
        delay_frame=0,
    )

    unet_dtype = _unet_dtype(unet)

    frame_idx = 0
    for whisper_batch, latent_batch in gen:
        audio_feature_batch = torch.from_numpy(whisper_batch).to(
            device=DEVICE, dtype=unet_dtype
        )
        audio_feature_batch = pe(audio_feature_batch)
        latent_batch = latent_batch.to(dtype=unet_dtype)

        pred_latents = _unet_forward(unet, latent_batch, audio_feature_batch)
        recon = _vae_decode(vae, pred_latents)

        for res_face in recon:
            cur_idx = _ping_pong_index(frame_idx, n_latents)
            ori_frame = frame_list[cur_idx].copy()
            bbox = coord_list[cur_idx]

            if _is_placeholder(bbox):
                combine = ori_frame
            else:
                x1, y1, x2, y2 = bbox
                res_face_resized = cv2.resize(
                    res_face.astype(np.uint8), (x2 - x1, y2 - y1)
                )
                combine = _blend(
                    ori_frame, res_face_resized, bbox,
                    mask_list[cur_idx], mask_coords_list[cur_idx],
                )

            ok, jpeg_buf = cv2.imencode(
                '.jpg', combine, [cv2.IMWRITE_JPEG_QUALITY, 85]
            )
            if not ok:
                raise RuntimeError(f'cv2.imencode failed at frame {frame_idx}')
            yield frame_idx, jpeg_buf.tobytes()
            frame_idx += 1

@app.websocket('/lipsync/stream')
async def lipsync_stream(ws: WebSocket):
    await ws.accept()
    try:
        first = json.loads(await ws.receive_text())
        avatar_id = first['avatar_id']

        pcm_buf = []
        while True:
            m = json.loads(await ws.receive_text())
            if m.get('type') == 'end':
                break
            if m.get('type') == 'pcm':
                pcm_buf.append(base64.b64decode(m['data']))
        full_pcm = b''.join(pcm_buf)
        print(f'[lipsync] received {len(full_pcm)} bytes PCM '
              f'(~{len(full_pcm)/2/24000:.1f}s audio)')

        for frame_idx, jpeg_bytes in _run_musetalk(avatar_id, full_pcm, fps=25):
            await ws.send_text(json.dumps({
                'type': 'frame',
                'frame_idx': frame_idx,
                'pts': frame_idx / 25.0,
                'jpeg_b64': base64.b64encode(jpeg_bytes).decode(),
            }))
        await ws.send_text(json.dumps({'type': 'end'}))
    except WebSocketDisconnect:
        pass
    except Exception as e:
        traceback.print_exc()
        try:
            await ws.send_text(json.dumps({'type': 'error', 'message': f'{type(e).__name__}: {e}'}))
        except Exception:
            pass

if __name__ == '__main__':
    import uvicorn
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

In [ ]:
# Cell 3 — Launch. Start (or restart) the Cloudflare tunnel + uvicorn.
#
# Pre-flight check first: verifies the py310 venv has uvicorn + cv2 + torch.
# If not, prints a clear "Cell 1 didn't finish" message and stops.
# Then kills any old uvicorn/cloudflared, starts fresh, and only prints READY
# after /healthz returns 200 — so a uvicorn crash never masquerades as success.
import subprocess, re, threading, time, os, sys
from collections import deque

# ─── Pre-flight ─────────────────────────────────────────────────────────────
print("=== Pre-flight check ===")
PY = '/content/py310/bin/python'
if not os.path.exists(PY):
    print(f"!! {PY} not found. Cell 1 hasn't been run yet — run Cell 1 first.")
    raise SystemExit
r = subprocess.run([PY, '-c',
    'import fastapi, uvicorn, cv2, torch; '
    'print("fastapi", fastapi.__version__); '
    'print("uvicorn", uvicorn.__version__); '
    'print("cv2", cv2.__version__); '
    'print("torch", torch.__version__, "cuda:", torch.cuda.is_available())'],
    capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print("!! Pre-flight FAILED:\n", r.stderr)
    print("Run Cell 1 again to fix missing packages.")
    raise SystemExit

if not os.path.exists('/content/server.py'):
    print("!! /content/server.py not found — run Cell 2 first.")
    raise SystemExit

print("✓ Pre-flight passed.\n")

# ─── Install cloudflared if missing ─────────────────────────────────────────
if not os.path.exists('/usr/local/bin/cloudflared'):
    print("Installing cloudflared...")
    subprocess.run(
        ['wget', '-q',
         'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
         '-O', '/usr/local/bin/cloudflared'],
        check=True,
    )
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

# ─── Kill any previous server / tunnel ──────────────────────────────────────
PORT = 8000
subprocess.run(f'fuser -k {PORT}/tcp 2>/dev/null', shell=True)
subprocess.run('pkill -f cloudflared 2>/dev/null', shell=True)
time.sleep(2)

# ─── Start tunnel + uvicorn in background threads ───────────────────────────
tunnel_url_ref = {'url': None}
uvicorn_tail   = deque(maxlen=40)

def run_tunnel():
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print('[tunnel]', line, end='')
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m and not tunnel_url_ref['url']:
            tunnel_url_ref['url'] = m.group(0)

def run_uvicorn():
    proc = subprocess.Popen(
        [PY, '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', str(PORT)],
        cwd='/content',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        uvicorn_tail.append(line.rstrip())
        print('[uvicorn]', line, end='')

threading.Thread(target=run_tunnel,  daemon=True).start()
threading.Thread(target=run_uvicorn, daemon=True).start()
print(f"Booting uvicorn (port {PORT}) and cloudflared quick-tunnel...")

# ─── Wait for tunnel URL ────────────────────────────────────────────────────
deadline = time.time() + 180
while time.time() < deadline and not tunnel_url_ref['url']:
    time.sleep(1)

if not tunnel_url_ref['url']:
    print("\n!! Tunnel URL not detected within 180s. Re-run this cell.")
else:
    # ─── Verify uvicorn is actually serving before claiming READY ──────────
    import urllib.request, json as _json
    healthz_ok = False
    last_err = None
    for _ in range(30):  # ~60 s
        try:
            req = urllib.request.Request(tunnel_url_ref['url'] + '/healthz')
            with urllib.request.urlopen(req, timeout=3) as r:
                _json.loads(r.read().decode())
                healthz_ok = True
                break
        except Exception as e:
            last_err = e
        time.sleep(2)

    if healthz_ok:
        print('\n' + '=' * 70)
        print(f'COLAB_INFERENCE_URL={tunnel_url_ref["url"]}')
        print('=' * 70)
        print('READY. Copy the line above into your local .env on Windows.')
    else:
        print('\n' + '!' * 70)
        print(f'Tunnel up at {tunnel_url_ref["url"]} BUT /healthz not responding.')
        print(f'Last error: {last_err}')
        print('Recent uvicorn output:')
        print('!' * 70)
        for line in uvicorn_tail:
            print(' ', line)